# How to Load an MPM-Toolbox Project (Score, Modelled Performance, Observed Alignment)

An MPM-Toolbox project is a triple of sibling XML files describing **one**
musical work: a `.msm` notated score, a `.mpm` *modelled* performance overlay
(tempo, dynamics, articulation, … as expressive markup), and a `.mpr` project
file carrying an *observed* audio-to-score alignment. This guide loads such a
project in a single call and arranges it as one multimodal
{{< glossary AlignmentBundle >}} spanning the logical (score) and physical
(performance) domains.

The work is Beethoven's *Eroica* Variations, Op. 35 — Var. XIV — in a 1971
Curzon recording. By the end we will have, in one bundle: the notated score in
two logical units, the performance markup carried as
{{< glossary Event >}}s, a modelled tempo curve mapping score quarters to
seconds, and the observed onsets tied back to the score note by note.

We **load** the modelled markup and the observed alignment exactly as they sit
on disk. Nothing here runs an aligner, and the tempo model is read as written
— we do not render a beat-accurate performance from it.

The arc:

1. Load the whole project in one call.
2. Read the logical score in two units, linked by a
   {{< glossary ConversionMap >}}.
3. Read the performance markup carried as {{< glossary Event >}}s on the score.
4. Read the modelled quarters→seconds tempo map.
5. Read the observed alignment — the physical performance and its cross-group
   {{< glossary MatchClaim >}}s.

## Setup

In [1]:
from __future__ import annotations

from timetoalign.core import TimeUnit
from timetoalign.loader.alignment import MpmLoader
from timetoalign.testdata import ensure_data

ROOT = ensure_data("mpm_toolbox")
MPR = (
    ROOT
    / "MPRproject_1971Curzon_VariationXIV"
    / "Beethoven_op35_1971Curzon_Var14only.mpr"
)

/home/laser/miniconda3/envs/timetoalign/lib/python3.11/site-packages/partitura/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 1. Load the project in one call

{{< glossary MpmLoader >}} is given the `.mpr` project file; it resolves the
sibling `.msm` and `.mpm` by the bare filenames the project names, parses the
score, the selected performance's markup, and the observed alignment, and
binds the recording's audio for its sample rate. `from_file()` is the one-line
form of the standard two-phase loader pattern.

In [2]:
loader = MpmLoader.from_file(MPR)
loader

Project,Beethoven_op35_1971Curzon_Var14only
Performance,MEI export performance
Claims,251
Timelines,"4 in 2 group(s) (score, perf)"
Create,"create_bundle(), create_timeline(), create_timelines()"


The `.mpm` may hold several `<performance>` blocks; by default the first is
selected. The pulses-per-quarter grid (720) and the chosen performance name
are read straight from the files. To select a different performance, pass
`MpmLoader().load(MPR, performance="...")`.

In [3]:
{
    "pulses per quarter": loader.ppq,
    "performance": loader.performance_name,
}

{'pulses per quarter': 720, 'performance': 'MEI export performance'}

`create_bundle()` assembles the {{< glossary AlignmentBundle >}}: four
timelines arranged in two groups — a shared logical `"score"` group and a
physical `"perf"` group. Everything below reads from this single bundle.

In [4]:
bundle = loader.create_bundle()
bundle

AlignmentBundle(id='bundle:AlignmentBundle_1', name='Beethoven_op35_1971Curzon_Var14only', timelines=4, groups=2)

In [5]:
{
    "timelines": bundle.n_timelines,
    "groups": bundle.n_groups,
    "groups_listed": bundle.group_ids,
    "timelines_listed": bundle.timeline_ids,
}

{'timelines': 4,
 'groups': 2,
 'groups_listed': ['score', 'perf'],
 'timelines_listed': ['score:clt1', 'score:dlt1', 'perf:cpt1', 'perf:dpt1']}

***

## 2. The logical score, in two units

The score lives in the `"score"` group, in **two logical units** — the same
notes, measured two ways:

- `score:dlt1` carries integer **tick** onsets (the `pulsesPerQuarter` grid the
  score was notated on).
- `score:clt1` carries **quarter-note** onsets (an exact rational).

In [6]:
score_dlt = bundle.get_timeline("score:dlt1")
score_dlt

DiscreteLogicalTimeline(id='score:dlt1', length=46440, unit=ticks, events=499, children=0, cmaps=1)

In [7]:
score_clt = bundle.get_timeline("score:clt1")
score_clt

ContinuousLogicalTimeline(id='score:clt1', length=46439/720, unit=quarters, events=251, children=0, cmaps=1)

Both hold the same 251 notes. The `.msm` records each note's MIDI pitch
together with the spelling it was notated with (`pitchname` / `octave`); a look
at the first few notes of `score:clt1` shows what the score carried:

In [8]:
score_clt.get_events().table.slice(0, 5).to_pandas()[
    ["id", "pitch", "pitchname", "octave"]
]

,id,pitch,pitchname,octave
0,nbwxzb1,75,e,4
1,n63xubs,82,b,4
2,n1hksdx3,62,d,3
3,n1vnikpu,65,f,3
4,n5hfy1h,70,b,3


### The tick grid and the quarter grid are one conversion apart

The two logical timelines are not independent: a `TicksToQuarters`
{{< glossary ConversionMap >}} on `score:dlt1` carries the tick grid to
quarters — 720 ticks to the quarter. A {{< glossary TimeStamp >}} on
`score:dlt1` exposes the conversion at any coordinate, so asking for the
quarter reading at a tick is the continuous↔discrete link of the logical
domain made visible:

In [9]:
{
    "tick 360 -> quarters": score_dlt.get_timestamp(360).get_unit(TimeUnit.quarters),
    "tick 720 -> quarters": score_dlt.get_timestamp(720).get_unit(TimeUnit.quarters),
    "tick 1440 -> quarters": score_dlt.get_timestamp(1440).get_unit(TimeUnit.quarters),
}

{'tick 360 -> quarters': 0.5,
 'tick 720 -> quarters': 1.0,
 'tick 1440 -> quarters': 2.0}

***

## 3. The performance markup, as events on the score

A `.mpm` overlays the score with expressive markup — tempo, dynamics,
articulation, asynchrony, and any other map type the project carries. The
loader places every markup entry on `score:dlt1` as an {{< glossary Event >}}
at its tick onset, sitting alongside the `Note` events. A single logical
timeline therefore carries both the notes and the modelled performance markup,
read with the same event query used everywhere else — filter by `event_type`.

The coordinate (`start`) comes back as a number; the remaining markup columns
currently round-trip as strings, so we cast them as we read.

### Tempo

Each `Tempo` event carries a beats-per-minute reading. The MPM value may be an
inline number or a **style name** declared in the performance's style block;
either way the loader resolves it to a number, keeping the original token in a
`*_label` column. Here the style name `"Meno mosso."` resolves to 100 BPM:

In [10]:
tempo_events = score_dlt.get_events().filter(event_type="Tempo").to_pandas()

tempo_table = tempo_events[["start", "bpm_label", "bpm"]].copy()
tempo_table["tick"] = tempo_table.pop("start").astype(int)
tempo_table["bpm"] = tempo_table["bpm"].astype(float)

tempo_table[["tick", "bpm_label", "bpm"]].head(7)

,tick,bpm_label,bpm
0,0,Meno mosso.,100.0
1,28440,25.0,25.0
2,28800,Meno mosso.,100.0
3,39960,25.0,25.0
4,40320,Meno mosso.,100.0
5,45000,25.0,25.0
6,46080,adagio,79.0


### Dynamics

`Dynamics` events resolve the same way: the dynamic mark `"p"` is a style name
that resolves to a MIDI volume of 48, with the mark preserved in
`volume_label`:

In [11]:
dynamics_events = score_dlt.get_events().filter(event_type="Dynamics").to_pandas()

dynamics_table = dynamics_events[["start", "volume_label", "volume"]].copy()
dynamics_table["tick"] = dynamics_table.pop("start").astype(int)
dynamics_table["volume"] = dynamics_table["volume"].astype(float)

dynamics_table[["tick", "volume_label", "volume"]].head(6)

,tick,volume_label,volume
0,0,p,48.0
1,6336,p,48.0
2,7560,f,72.0
3,9000,f,72.0
4,10440,p,48.0
5,17640,p,48.0


### Articulation

`Articulation` events name an articulation (`staccato`, `tenuto`, …) and, where
the performance declares a definition for it, resolve its numeric attributes.
A `staccato` here resolves to an absolute played duration of 160 ms; the
`noteid` it applies to is carried alongside (its leading `#` stripped):

In [12]:
articulation_events = (
    score_dlt.get_events().filter(event_type="Articulation").to_pandas()
)

staccato = articulation_events[articulation_events["name"] == "staccato"].copy()
staccato["tick"] = staccato.pop("start").astype(int)
staccato["absolute_duration_ms"] = staccato["absolute_duration_ms"].astype(float)

staccato[["tick", "name", "absolute_duration_ms", "noteid"]].head(5)

,tick,name,absolute_duration_ms,noteid
173,29160,staccato,160.0,ngx1f26
174,29160,staccato,160.0,n14ahlx1
175,30600,staccato,160.0,nbeyber
176,30600,staccato,160.0,n1y3yzsw
177,32040,staccato,160.0,ncgjw5z


The point of this section: tempo and dynamics style names resolve to numeric
values, articulations resolve to played durations and velocities, and *every*
one of them is an event on the score timeline, queried the same way the notes
are. Any map type the loader does not model specially is still emitted, with
its raw attributes carried verbatim, so nothing in a project is silently
dropped.

***

## 4. The modelled tempo, as quarters → seconds

The performance's tempo markup also defines, segment by segment, how fast the
score is taken. The loader integrates it into a modelled quarters→seconds
{{< glossary ConversionMap >}} — a `TableMap` it exposes directly:

In [13]:
tempo_map = loader.tempo_map
tempo_map

TableMap(n_points=8, source_unit=quarters, target_unit=seconds, kind=linear)

It is anchored at the score's start (`0` quarters → `0` seconds) and walks
forward at each tempo segment's pace. Converting a couple of quarter positions
reads the modelled clock-time at which the score reaches them:

In [14]:
{
    "quarter 0 -> seconds": tempo_map(0),
    "quarter 39.5 -> seconds": tempo_map(39.5),
    "quarter 40.0 -> seconds": tempo_map(40.0),
}

{'quarter 0 -> seconds': 0.0,
 'quarter 39.5 -> seconds': 23.7,
 'quarter 40.0 -> seconds': 24.9}

This is a **constant-tempo-per-segment** model: each tempo entry sets a flat
pace until the next. Accelerando / ritardando ramps (an entry's
`transition.to`) are preserved as a `Tempo`-event attribute but are *not*
rendered into the curve — the map reads what the project modelled, deliberately
stopping short of synthesising a beat-accurate performance.

***

## 5. The observed alignment

The model above is one account of the performance. The `.mpr` carries another:
an **observed** alignment, recording for every score note the moment it was
actually played in the recording. The loader places these onsets in the
physical `"perf"` group, in **two units** — `perf:cpt1` in seconds and
`perf:dpt1` in samples, linked (as in any physical timeline) by a
`SamplesToSeconds` {{< glossary ConversionMap >}} carrying the recording's
sample rate.

In [15]:
perf_cpt = bundle.get_timeline("perf:cpt1")
perf_cpt

ContinuousPhysicalTimeline(id='perf:cpt1', length=75.37598985608956, unit=seconds, events=251, children=0)

The score group and the performance group are tied together by cross-group
{{< glossary MatchClaim >}}s — one per score note. The `.mpr` alignment is a
perfect bijection (every score note has exactly one observed onset and vice
versa), so every claim is **synchronous**: there are no
{{< glossary NOMATCH >}} gaps.

In [16]:
claims = bundle.cross_group_claims

{
    "total claims": len(claims),
    "synchronous (matched)": sum(1 for c in claims if c.is_synchronous),
    "NOMATCH": sum(1 for c in claims if not c.is_synchronous),
}

{'total claims': 251, 'synchronous (matched)': 251, 'NOMATCH': 0}

A single claim relates a score quarter to an observed performed second. This
one anchors the note at score quarter 0.5 to the moment it was played, 0.8
seconds into the recording:

In [17]:
quarter_half_claims = [
    c
    for c in claims
    if c.is_synchronous
    and c.start_anchor is not None
    and c.start_anchor.coordinate_a == 0.5
]
quarter_half_claims[0]

MatchClaim(instant: score:clt1@0.5 <-> perf:cpt1@0.8)

### One score position, read across the domains

Because the performance is anchored back to the score, a single score
coordinate resolves across the whole bundle. `get_matchstamp_at` takes a
coordinate on `score:clt1` (quarters) and returns the corresponding coordinate
on every connected timeline — here score quarter 2.0 mapped to the second at
which it was observed:

In [18]:
stamp = bundle.get_matchstamp_at(2.0, "score:clt1")
stamp

ID,Coordinate,Type
score:clt1,2,anchor
perf:cpt1,2.177196,anchor


Read across that {{< glossary MatchStamp >}}: the same notated quarter resolves
to a logical position (2.0 quarters) and an observed physical position (≈ 2.18
seconds), one query crossing from the score domain to the performance domain
with no aligner ever run.

## Recap

| What the bundle expresses | How |
|---|---|
| Score, two logical units | `score:dlt1` (ticks) + `score:clt1` (quarters), one `TicksToQuarters` map |
| Performance markup | `Tempo` / `Dynamics` / `Articulation` events on `score:dlt1`, via `filter(event_type=...)` |
| Modelled tempo | a quarters→seconds `TableMap` (`loader.tempo_map`), constant-tempo-per-segment |
| Observed performance, two physical units | `perf:cpt1` (s) + `perf:dpt1` (samples), one `SamplesToSeconds` |
| Score ↔ performance | one synchronous {{< glossary MatchClaim >}} per note (a perfect bijection) |
| A score position read across domains | `bundle.get_matchstamp_at(coord, "score:clt1")` |

One {{< glossary AlignmentBundle >}} carries the notated score (in ticks and
quarters, with its modelled performance markup and tempo) and the observed
performance (in seconds and samples), linked note by note — a single object
holding what was notated, how it was modelled, and how it was actually played.